# Inspect Aggregated AET Datasets

HRU-level inspection of the three AET sources. Mirrors `inspect_consolidated_aet.ipynb`.

Sources (see `catalog/variables.yml` → `aet`):

- SSEBop (`et`, mm/month) — accessed via USGS NHGF STAC; consolidated and aggregated.
- MOD16A2 v061 (`ET_500m`, kg m⁻² per 8-day composite — `scale_factor=0.1`) — global tiles consolidated to CONUS+ then aggregated.
- MWBM (Wieczorek et al. 2024, `aet`, mm/month) — CONUS-wide MWBM forced by ClimGrid; aggregated to HRU polygons per year. Native mm/month.

The AET target combines these via per-HRU per-month **min/max in absolute mm/month** (`multi_source_minmax`, not 0–1 normalised) — so cross-source magnitudes feed the target bound directly. That makes calendar-month cadence the right comparison axis: SSEBop and MWBM are already mm/month, but MOD16A2's 46-per-year 8-day composites have to be resampled to 12-per-year monthly totals via overlap weighting before any side-by-side plot is meaningful (recipes §2). The notebook does that conversion in the helper defined below and reuses it through the histogram, time-series, and validation cells.

See `docs/references/calibration-target-recipes.md` §2 for canonical-unit conversions and the open `scale_factor` question.

## Per-target conventions in this notebook

- SSEBop `et` is native mm/month. No conversion before plotting or comparison.
- MOD16A2 `ET_500m` is kg m⁻² per 8-day composite, with `scale_factor=0.1` per the HDF spec. We apply the scale factor explicitly in `_mod16a2_to_monthly_mm` (defined below) to defend against the missing-`decode_cf` case (recipes §2 — flagged 500× domain-mean offset in the consolidated notebook).
- MWBM `aet` is native mm/month. No conversion.
- **Per-month aggregation of MOD16A2 8-day composites.** Each composite covers an 8-day window starting at its timestamp; the year-end composite at DOY 361 is shorter (5–6 days) because LP DAAC caps it at the calendar-year boundary. For each calendar month, weight every composite that intersects the month by `(days of overlap with month) / composite_length`, sum the weighted contributions, and multiply by `scale_factor`. The result is mm/month at the same cadence as SSEBop and MWBM, and matches what `targets/aet.py` will compute (recipes §2). Earlier versions of this notebook used a "pick one composite, label it mm/month" shortcut that under-counted MOD16A2 by ~3–4× — that shortcut is no longer used because the AET target consumes absolute monthly totals, not 8-day samples, in a multi-source min/max.
- Reference source: SSEBop.

In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/inspect_aggregated/<run>_*.png
import _helpers
_helpers.SAVE_FIGURES = True

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "aet"
TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000
TARGET_MONTH = 1
TIME_SERIES_YEARS = range(2000, 2011)
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

SOURCES = {
    "ssebop": {"label": "SSEBop (et)", "var": "et"},
    "mod16a2_v061": {"label": "MOD16A2 v061 (ET_500m)", "var": "ET_500m"},
    "mwbm_climgrid": {"label": "MWBM (ClimGrid, aet)", "var": "aet"},
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")


## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key)
    if paths is None:
        print(f"SKIP {info['label']}: no aggregated NCs at "
              f"{project_dir}/data/aggregated/{source_key}/")
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR)
    info["units"] = unit_from_catalog(source_key, info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get(fabric_cfg['id_col'], 'unknown')} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )


In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)


In [ ]:
if not opened:
    print("No sources available; skipping native-unit map.")
else:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            cmap="YlGnBu",
            title=f"{info['label']}\n{TARGET_TIME} | {info['units']}",
            units=info["units"],
        )
    fig.suptitle(f"AET — native units, {TARGET_TIME}", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()


## Calendar-month comparison map (mm/month)

Convert each source to **mm/month** and plot side by side on a shared colour scale derived from SSEBop (the reference source). All three are at the same calendar-month cadence after this step:

- SSEBop `et` — native mm/month, pass-through.
- MOD16A2 v061 `ET_500m` — overlap-weighted sum of 8-day composites that intersect the target month, × `scale_factor = 0.1`. `_mod16a2_to_monthly_mm` does the resample; `_to_monthly_mm` dispatches per source. Mirrors `targets/aet.py` (recipes §2).
- MWBM `aet` — native mm/month, pass-through.

All three panels are on the HRU fabric, so the geographic footprint is identical; the colour scale is anchored on SSEBop's distribution. Because the AET target uses absolute mm/month values directly (`multi_source_minmax` with no 0–1 normalisation), this is the figure where cross-source magnitude differences are physically meaningful — they propagate straight into the per-HRU min/max bound.

In [ ]:
def _mod16a2_to_monthly_mm(
    da: xr.DataArray,
    scale_factor: float = 0.1,
    composite_days: int = 8,
) -> xr.DataArray:
    """Resample MOD16A2 8-day composite kg m⁻² to calendar-month mm totals.

    For each calendar month spanned by ``da.time``, sum overlap-weighted
    contributions of every composite that intersects the month:

        month_mm = scale_factor * Σ_c (composite_c * overlap_days_c / composite_length_c)

    The standard composite covers 8 days; the year-end composite (DOY 361)
    is capped at the next calendar year's Jan 1 by LP DAAC and covers 5-6
    days — the cap prevents day-1-of-Jan double counting between Dec 26's
    nominal-8-day window and Jan 1's actual composite. Mirrors what
    targets/aet.py should compute (recipes §2).
    """
    starts = pd.DatetimeIndex(da.time.values)
    year_ends = pd.DatetimeIndex(
        [pd.Timestamp(year=t.year + 1, month=1, day=1) for t in starts]
    )
    nominal_ends = starts + pd.Timedelta(days=composite_days)
    ends = pd.DatetimeIndex(np.minimum(nominal_ends.values, year_ends.values))
    composite_lengths = (ends - starts).days.to_numpy().astype(float)

    first_month = pd.Timestamp(year=starts[0].year, month=starts[0].month, day=1)
    last_dt = ends[-1] - pd.Timedelta(seconds=1)
    last_month = pd.Timestamp(year=last_dt.year, month=last_dt.month, day=1)
    candidate_starts = pd.date_range(first_month, last_month, freq="MS")
    candidate_ends = candidate_starts + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)

    full = (candidate_starts >= starts[0]) & (candidate_ends <= ends[-1])
    month_starts = candidate_starts[full]
    month_ends = candidate_ends[full]

    overlap_starts = np.maximum(starts.values[None, :], month_starts.values[:, None])
    overlap_ends = np.minimum(ends.values[None, :], month_ends.values[:, None])
    overlap_days = np.clip(
        (overlap_ends - overlap_starts) / np.timedelta64(1, "D"),
        0,
        None,
    )
    weights = overlap_days / composite_lengths[None, :]

    weight_da = xr.DataArray(
        weights,
        coords={"month": month_starts, "time": starts},
        dims=["month", "time"],
    )
    monthly = xr.dot(weight_da, da, dims="time") * scale_factor
    return monthly.rename({"month": "time"})


def _to_monthly_mm(da: xr.DataArray, source_key: str) -> xr.DataArray:
    """Dispatch a (time, ...) DataArray to mm/month per spatial unit.

    SSEBop and MWBM are already mm/month — pass through. MOD16A2 is
    8-day kg m⁻² composites — resample with overlap weighting.
    """
    if source_key in ("ssebop", "mwbm_climgrid"):
        return da
    if source_key == "mod16a2_v061":
        return _mod16a2_to_monthly_mm(da)
    raise ValueError(f"No monthly conversion for {source_key}")


if opened:
    converted = {}
    for source_key, (ds, info) in opened.items():
        monthly = _to_monthly_mm(ds[info["var"]], source_key)
        converted[source_key] = select_month(monthly, TARGET_YEAR, TARGET_MONTH).to_pandas()

    ref_key = "ssebop"
    if ref_key in converted:
        ref_vals = converted[ref_key].dropna().values
        vmin, vmax = float(np.percentile(ref_vals, 2)), float(np.percentile(ref_vals, 98))
    else:
        all_vals = np.concatenate([s.dropna().values for s in converted.values()])
        vmin, vmax = float(np.percentile(all_vals, 2)), float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(1, len(converted), figsize=(8 * len(converted), 6), squeeze=False)
    for idx, (source_key, values) in enumerate(converted.items()):
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            vmin=vmin,
            vmax=vmax,
            cmap="YlGnBu",
            title=f"{SOURCES[source_key]['label']}\n{TARGET_TIME} | mm/month",
            units="mm/month",
        )
    fig.suptitle(
        f"AET — mm/month, colour scale from SSEBop — {TARGET_TIME}",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()


In [ ]:
if opened:
    fig, ax = plt.subplots(figsize=(10, 5))
    for source_key, values in converted.items():
        ax.hist(
            values.dropna(),
            bins=60,
            histtype="step",
            label=SOURCES[source_key]["label"],
            density=True,
            linewidth=2,
        )
    ax.set_xlabel("AET (mm/month)")
    ax.set_ylabel("Density")
    ax.set_title(f"Cross-source HRU distribution — {TARGET_TIME}")
    ax.legend()
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()


In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    series_data = {}
    for source_key, info in SOURCES.items():
        if source_key not in opened:
            continue
        ds_range = open_year_range(project_dir, source_key, TIME_SERIES_YEARS)
        try:
            id_dim = fabric_cfg["id_col"]
            sel = ds_range[info["var"]].sel({id_dim: list(rep_hrus.values())}).load()
        finally:
            ds_range.close()
        # Resample to monthly mm so all sources share calendar-month cadence.
        # MOD16A2's 46/year 8-day composites become 12/year monthly totals
        # via the same overlap-weighted sum the AET target builder uses.
        series_data[source_key] = _to_monthly_mm(sel, source_key)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    id_dim = fabric_cfg["id_col"]
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        for source_key, da in series_data.items():
            da_hru = da.sel({id_dim: hru_id})
            ax.plot(da_hru.time, da_hru.values, label=SOURCES[source_key]["label"])
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("mm/month")
        ax.legend(fontsize=8)
    fig.suptitle(f"AET at representative HRUs — {min(TIME_SERIES_YEARS)}–{max(TIME_SERIES_YEARS)}")
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()


In [ ]:
if opened:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        n_nan = nan_hru_count(values)
        print(f"{info['label']}: {n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)")
        plot_nan_hrus(
            axes[0, idx],
            fabric,
            values,
            title=f"{info['label']}\nNaN HRUs (red) — {n_nan} of {len(fabric)}",
        )
    fig.suptitle(
        "Coverage gaps — these will be nearest-neighbor-filled in normalize/ before target combination",
        fontsize=12,
        y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()


In [ ]:
def _mod16a2_target_month_gridded_mm(ds, var, target_year, target_month, composite_days=8):
    """Compute mm/month for the gridded NC of a single target month.

    Subsets the dataset to composites that intersect the target month
    before .load(), then delegates to _mod16a2_to_monthly_mm. Avoids
    pulling the full year of CONUS+ 500 m grid (~9 GB) into memory just
    to read one month for the validation cross-check below.
    """
    da = ds[var]
    starts = pd.DatetimeIndex(da.time.values)
    target_start = pd.Timestamp(target_year, target_month, 1)
    target_end = target_start + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)

    year_ends = pd.DatetimeIndex(
        [pd.Timestamp(year=t.year + 1, month=1, day=1) for t in starts]
    )
    ends = pd.DatetimeIndex(
        np.minimum(
            (starts + pd.Timedelta(days=composite_days)).values,
            year_ends.values,
        )
    )

    mask = (starts < target_end) & (ends > target_start)
    if not mask.any():
        raise ValueError(f"No MOD16A2 composites in {target_year}-{target_month:02d}")

    sub = da.isel(time=np.where(mask)[0]).load()
    monthly = _mod16a2_to_monthly_mm(sub)
    return select_month(monthly, target_year, target_month)


def _gridded_mean_aet(source_key, info):
    """Compute the gridded CONUS-footprint mean for the validation cell.

    Approximate cross-check, not a like-for-like comparison: gridded mean
    is unweighted over the consolidated-NC bbox (which extends past the
    fabric into ocean/Canada/Mexico edges); HRU mean is Albers-area-
    weighted over fabric HRUs only. Differences of 5-30% are normal for
    sources with significant out-of-fabric coverage.

    For MOD16A2 we use the same overlap-weighted monthly resample as the
    aggregated path so the two columns of the table compare like-for-like
    monthly totals (not an 8-day sample vs. a calendar-month sum).
    """
    if source_key == "ssebop":
        # SSEBop is remote zarr accessed via the NHGF STAC.
        # If the project's network access doesn't allow STAC reads at notebook
        # time, skip-with-reason.
        try:
            import fsspec
            store = "s3://mdmf/gdp/ssebopeta_monthly.zarr/"
            ds = xr.open_zarr(
                fsspec.get_mapper(
                    store,
                    anon=True,
                    client_kwargs={"endpoint_url": "https://usgs.osn.mghpcc.org/"},
                ),
                consolidated=True,
            )
        except Exception as exc:
            return None, f"STAC fetch failed: {exc}"
        try:
            da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH).load()
        finally:
            ds.close()
        return da.mean(skipna=True).item(), None
    if source_key == "mod16a2_v061":
        path = datastore_dir / "mod16a2_v061" / f"mod16a2_v061_{TARGET_YEAR}_consolidated.nc"
        if not path.exists():
            return None, f"missing consolidated NC at {path}"
        with xr.open_dataset(path) as ds:
            month_da = _mod16a2_target_month_gridded_mm(
                ds, info["var"], TARGET_YEAR, TARGET_MONTH
            )
        return float(month_da.mean(skipna=True).item()), None
    if source_key == "mwbm_climgrid":
        # Single CF NetCDF holding the full 1895-2020 monthly history at
        # ~0.042° CONUS. Slicing one month before .load() keeps the
        # validation footprint small (~7 MB).
        path = datastore_dir / "mwbm_climgrid" / "ClimGrid_WBM.nc"
        if not path.exists():
            return None, f"missing consolidated NC at {path}"
        with xr.open_dataset(path) as ds:
            da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH).load()
        return float(da.mean(skipna=True).item()), None
    return None, f"unknown gridded path for {source_key}"


print(f"{'Source':<35} {'Aggregated':>12} {'Gridded':>12} {'Δ':>12} {'% diff':>8}")
print("-" * 85)
for source_key, (ds, info) in opened.items():
    monthly = _to_monthly_mm(ds[info["var"]], source_key)
    converted_agg = select_month(monthly, TARGET_YEAR, TARGET_MONTH).to_pandas()
    agg_mean = area_weighted_mean(converted_agg, fabric)
    gridded_mean, reason = _gridded_mean_aet(source_key, info)
    if gridded_mean is None:
        print(f"{info['label']:<35} {agg_mean:>12.3f}  {'SKIP':>12} ({reason})")
        continue
    diff = agg_mean - gridded_mean
    pct = 100 * diff / gridded_mean if gridded_mean else float("nan")
    print(f"{info['label']:<35} {agg_mean:>12.3f} {gridded_mean:>12.3f} {diff:>12.3f} {pct:>7.2f}%")


## Why HRU-level patterns differ across sources

After area-weighted aggregation to HRUs (and, for MOD16A2, overlap-weighted resample from 8-day composites to mm/month), the HRU-level magnitudes are within the same order of magnitude as the gridded means (validation cell above). The validation cell's "gridded" column is an unweighted mean over the consolidated-NC bbox, which extends past the fabric into ocean and Canada/Mexico edges; the "aggregated" column is an Albers-area-weighted mean over fabric HRUs only. A 5–30% difference between the two columns is normal for sources with out-of-fabric coverage. Differences above and beyond that geographic mismatch are physical:

- **SSEBop** is energy-balance-derived from satellite LST and meteorological forcing. It captures water stress directly through the surface energy balance. Native mm/month at 1 km.
- **MOD16A2** is the MODIS Penman-Monteith global ET product (VIIRS/MODIS LAI and surface reflectance with reanalysis forcing) at 500 m, 8-day composite cadence. The 8-day cadence smooths sub-monthly variability that SSEBop captures; aggregation to mm/month here uses overlap-weighted summation of the composites that intersect each calendar month.
- **MWBM (ClimGrid forcing)** is a process-modelled monthly water balance forced by gridded climate data — conceptually different from the two satellite-derived sources, which is exactly why it adds useful spread to the multi-source range.

**Calibration target implication.** The AET target uses all three sources as a per-HRU per-month `min/max` over absolute mm/month — there is no 0–1 normalisation, so the inter-source magnitudes here propagate straight into the calibration bound. Two gotchas to watch: the MOD16A2 `scale_factor=0.1` (without it, MOD16A2 reads 500× too high and the multi-source range becomes degenerate), and the 8-day → monthly resample (without it, MOD16A2 reads ~3–4× too low because a single 8-day composite is being labelled as a month).

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()
